# 02 — Data Understanding and Cleaning

## Description

This notebook inspects and cleans the raw datasets.

The goal is to move from raw CSV files to clean, documented, analysis-ready datasets that can be used in the next notebooks for analytical modeling, feature engineering, data analysis and dashboard preparation.

Datasets used:

- `DataCoSupplyChainDataset.csv` — main order, customer, product, sales, logistics, and delivery dataset.
- `tokenized_access_logs.csv` — web access/clickstream dataset containing product/category page views.

Main tasks covered:

- load both raw CSV files
- inspect shape, schema, and column names
- standardize column names
- check missing values and duplicates
- parse date/time fields
- validate important business fields
- conduct cleaning and create quality-control features
- export cleaned datasets for downstream notebooks

## Table of Contents
1. Notebook Setup
2. Load Raw Datasets

3. Initial Dataset Exploration
   3.1 Dataset Dimensions
   3.2 Schema Summary
   3.3 Preview
   3.4 Initial Observations

4. Grain and Entity Understanding
   4.1 Transactions Dataset Grain
   4.2 Logs Dataset Grain
   4.3 Relationship Between Datasets
   4.4 Summary

5. Data Quality Assessment
   5.1 Missing Values
   5.2 Constant Columns
   5.3 Redundant Columns
   5.4 Key Validation
   5.5 Category Mapping Validation

6. Cleaning Decisions & Transformations
   6.1 Standardize Column Names
   6.2 Drop Columns
   6.3 Convert Data Types
   6.4 Final Validation

7. Save Cleaned Datasets

## 1. Notebook Setup

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

RAW_TRANSACTIONS_PATH = RAW_DIR / 'DataCoSupplyChainDataset.csv'
RAW_LOGS_PATH = RAW_DIR / 'tokenized_access_logs.csv'

CLEAN_TRANSACTIONS_PATH = PROCESSED_DIR / 'cleaned_supply_chain.csv'
CLEAN_LOGS_PATH = PROCESSED_DIR / 'cleaned_access_logs.csv'

PARQUET_TRANSACTIONS_PATH = PROCESSED_DIR / 'cleaned_supply_chain.parquet'
PARQUET_LOGS_PATH = PROCESSED_DIR / 'cleaned_access_logs.parquet'

## 2. Load Raw Datasets

In [4]:
transactions_raw = pd.read_csv(RAW_TRANSACTIONS_PATH, encoding='latin1')
logs_raw = pd.read_csv(RAW_LOGS_PATH)

## 3. Initial Dataset Exploration

### 3.1 Datasets Dimensions

In [5]:
dataset_shapes = pd.DataFrame({
    "dataset": ["transactions", "logs"],
    "rows": [transactions_raw.shape[0], logs_raw.shape[0]],
    "columns": [transactions_raw.shape[1], logs_raw.shape[1]]
})

dataset_shapes

,dataset,rows,columns
0,transactions,180519,53
1,logs,469977,8


### 3.2 Schema summary

In [6]:
def dataset_summary(df: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame({
            "column_name": df.columns,
            "dtype": df.dtypes.astype(str),
            "non_null_count": df.notna().sum().values,
            "null_count": df.isna().sum().values,
            "unique_values": df.nunique().values,
        })
        .sort_values("null_count", ascending=False)
        .reset_index(drop=True)
    )

In [7]:
dataset_summary(transactions_raw)

,column_name,dtype,non_null_count,null_count,unique_values
0,Product Description,float64,0,180519,0
1,Order Zipcode,float64,24840,155679,609
2,Customer Lname,str,180511,8,1109
3,Customer Zipcode,float64,180516,3,995
4,Days for shipment (scheduled),int64,180519,0,4
5,Sales per customer,float64,180519,0,2927
6,Benefit per order,float64,180519,0,21998
7,Delivery Status,str,180519,0,4
8,Late_delivery_risk,int64,180519,0,2
9,Customer City,str,180519,0,563


In [8]:
dataset_summary(logs_raw)

,column_name,dtype,non_null_count,null_count,unique_values
0,Product,str,469977,0,76
1,Category,str,469977,0,33
2,Date,str,469977,0,160815
3,Month,str,469977,0,5
4,Hour,int64,469977,0,24
5,Department,str,469977,0,6
6,ip,str,469977,0,3340
7,url,str,469977,0,152


### 3.3 Preview

In [9]:
transactions_raw.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [10]:
logs_raw.head(3)

,Product,Category,Date,Month,Hour,Department,ip,url
0,adidas Brazuca 2017 Official Match Ball,baseball & softball,9/1/2017 6:00,Sep,6,fitness,37.97.182.65,/department/fitness/category/baseball%20&%20so...
1,The North Face Women's Recon Backpack,hunting & shooting,9/1/2017 6:00,Sep,6,fan shop,206.56.112.1,/department/fan%20shop/category/hunting%20&%20...
2,adidas Kids' RG III Mid Football Cleat,featured shops,9/1/2017 6:00,Sep,6,apparel,215.143.180.0,/department/apparel/category/featured%20shops/...


### 3.4 Initial Observations
- The main dataset contains 180,519 rows and 53 columns. The logs dataset contains 469977 rows and 8 columns.

- The main dataset combines customer, product, order, financial, and shipping attributes in a single denormalized table. Several columns appear to require further investigation, including fields with constant values, fully missing values, high missingness, and potentially redundant identifiers.

## 4. Grain and Entity Understanding
Before assessing data quality or applying transformations, it is necessary to determine the granularity of each dataset and identify the primary business entities represented.

Understanding the grain ensures that subsequent metrics, aggregations, and analyses are calculated correctly.

### 4.1 Transactions Dataset Grain


In [11]:
grain_summary = pd.DataFrame({
    "metric": [
        "Total Rows",
        "Unique Orders",
        "Unique Order Items",
        "Unique Customers",
        "Unique Products"
    ],
    "value": [
        len(transactions_raw),
        transactions_raw["Order Id"].nunique(),
        transactions_raw["Order Item Id"].nunique(),
        transactions_raw["Customer Id"].nunique(),
        transactions_raw["Product Card Id"].nunique()
    ]
})

grain_summary

,metric,value
0,Total Rows,180519
1,Unique Orders,65752
2,Unique Order Items,180519
3,Unique Customers,20652
4,Unique Products,118


In [12]:
order_item_counts = (
    transactions_raw
    .groupby("Order Id")["Order Item Id"]
    .count()
    .describe()
)

order_item_counts

count    65752.000000
mean         2.745453
std          1.478363
min          1.000000
25%          1.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: Order Item Id, dtype: float64

In [13]:
order_customer_counts = (
    transactions_raw
    .groupby("Order Id")["Customer Id"]
    .nunique()
    .value_counts()
)
order_customer_counts

Customer Id
1    65752
Name: count, dtype: int64

### 4.2 Logs Dataset Grain

In [14]:
logs_grain_summary = pd.DataFrame({
    "metric": [
        "Total Rows",
        "Unique Products",
        "Unique Categories",
        "Unique Departments",
        "Unique IP Addresses",
        "Unique URLs"
    ],
    "value": [
        len(logs_raw),
        logs_raw["Product"].nunique(),
        logs_raw["Category"].nunique(),
        logs_raw["Department"].nunique(),
        logs_raw["ip"].nunique(),
        logs_raw["url"].nunique()
    ]
})

logs_grain_summary

,metric,value
0,Total Rows,469977
1,Unique Products,76
2,Unique Categories,33
3,Unique Departments,6
4,Unique IP Addresses,3340
5,Unique URLs,152


### 4.3 Relationship Between Datasets
This section checks whether the transactional dataset and the access logs dataset can be connected through shared product information.

Product name is the only available candidate for linking both datasets.

In [15]:
transaction_products = set(transactions_raw["Product Name"])
log_products = set(logs_raw["Product"])

shared_products = log_products & transaction_products
logs_only_products = log_products - transaction_products
transactions_only_products = transaction_products - log_products

product_overlap_summary = pd.DataFrame({
    "metric": [
        "Unique products in transactions",
        "Unique products in logs",
        "Shared products",
        "Products only in logs",
        "Products only in transactions",
        "Are all log products in transactions?"
    ],
    "value": [
        len(transaction_products),
        len(log_products),
        len(shared_products),
        len(logs_only_products),
        len(transactions_only_products),
        log_products.issubset(transaction_products)
    ]
})

product_overlap_summary

,metric,value
0,Unique products in transactions,118
1,Unique products in logs,76
2,Shared products,72
3,Products only in logs,4
4,Products only in transactions,46
5,Are all log products in transactions?,False


In [16]:
sorted(logs_only_products)

['TaylorMade 2017 Purelite Stand Bag',
 "Top Flite Women's 2017 XL Hybrid",
 'adidas Brazuca 2017 Official Match Ball',
 "adidas Kids' RG III Mid Football Cleat"]

In [17]:
sorted(transactions_only_products)

['Adult dog supplies',
 'Baby sweater',
 'Bowflex SelectTech 1090 Dumbbells',
 "Brooks Women's Ghost 6 Running Shoe",
 'CDs of rock',
 "Children's heaters",
 'DVDs ',
 'Dell Laptop',
 "Diamondback Girls' Clarity 24 Hybrid Bike 201",
 'ENO Atlas Hammock Straps',
 'Elevation Training Mask 2.0',
 'Fighting video games',
 'First aid kit',
 "Glove It Women's Mod Oval 3-Zip Carry All Gol",
 'GoPro HERO3+ Black Edition Camera',
 'GolfBuddy VT3 GPS Watch',
 "Hirzl Women's Hybrid Golf Glove",
 'Industrial consumer electronics',
 "LIJA Women's Argyle Golf Polo",
 'Lawn mower',
 "Men's gala suit",
 "Merrell Women's Grassbow Sport Waterproof Hik",
 "Nike Kids' Grade School KD VI Basketball Shoe",
 "Nike Men's Deutschland Weltmeister Winners Bl",
 "Nike Men's Kobe IX Elite Low Basketball Shoe",
 "Nike Women's Legend V-Neck T-Shirt",
 "Nike Women's Tempo Shorts",
 'Pelican Maverick 100X Kayak',
 'Porcelain crafts',
 'Rock music',
 'Smart watch ',
 'Sports Books ',
 'Summer dresses',
 'TaylorMade 201

In [18]:
len(shared_products) / len(log_products)

0.9473684210526315

### 4.4 Summary

The transactional dataset is stored at the **order-item grain**, where each row represents a single product line within an order. Orders act as the parent entity, may contain multiple products, and are associated with exactly one customer. This grain will be used throughout the project to ensure that aggregations and business metrics are calculated correctly.

The logs dataset captures **website browsing activity** and appears to be stored at the event level, where each row represents a single website access associated with a product, category, department, URL, and timestamp. Due to the absence of customer, order, and session identifiers, the dataset cannot support user-level or order-level analysis.

A comparison of product catalogs identified substantial overlap between the two datasets, with approximately 95% of products in the logs dataset also appearing in the transactional dataset. However, differences in product coverage and naming prevent a reliable direct merge. Therefore, the transactional dataset will serve as the primary analytical source, while the logs dataset will be used as a supplementary source for product interest and browsing behavior.